pip install pandas openpyxl matplotlib scipy

In [1]:
import os
import re
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

In [ ]:
# ------------------------------------------------------------
# CONFIG — file paths and display constants only.
# No vulnerability category / OWASP / severity is hardcoded here;
# those are read from the two Excel documents (see the data-prep cell).
# ------------------------------------------------------------
MASTER_FILE = "All_Application_Findings"   # the master detection matrix
MASTER_SHEET = "Findings"
MAPPING_FILE = "Vulnerability_Mapping"       # the reviewed CWE/OWASP/CVSS mapping
MAPPING_SHEET = "Mapping"
OUT_DIR = "figures"

TOOLS = ["Scanner", "Semgrep", "Snyk", "ZAP"]      # tool columns (Y/N) in the master sheet
TOOL_LABELS = {
    "Scanner": "Platform scanner",
    "Semgrep": "Semgrep (SAST)",
    "Snyk":    "Snyk (SCA)",
    "ZAP":     "OWASP ZAP (DAST)",
}

BOLT_COLOUR = "#2B59C3"
LOVABLE_COLOUR = "#D1495B"

SEVERITY_ORDER = ["Critical", "High", "Medium", "Low"]
SEVERITY_COLOURS = {"Critical": "#7F1D1D", "High": "#E4572E",
                    "Medium": "#F3A712", "Low": "#9DB4C0"}

OWASP_NAMES = {
    "A01": "A01 Broken Access Control",
    "A02": "A02 Security Misconfiguration",
    "A03": "A03 Supply Chain Failures",
    "A04": "A04 Cryptographic Failures",
    "A05": "A05 Injection",
    "A06": "A06 Insecure Design",
    "A07": "A07 Authentication Failures",
    "A08": "A08 Integrity Failures",
    "A09": "A09 Logging & Alerting",
    "A10": "A10 Exceptional Conditions",
}


In [ ]:
df = pd.read_excel(MASTER_FILE)

df.head(10) 

In [ ]:
df = pd.read_excel(MAPPING_FILE)

df.head(10) 

In [8]:
# Standard Category is taken verbatim from the master sheet — the single source of truth.
def clean_category(value):
    return str(value).strip()


In [9]:
# Build the OWASP lookup from the reviewed mapping document (one row per category).
mapping = pd.read_excel(MAPPING_FILE, sheet_name=MAPPING_SHEET)
mapping.columns = [str(c).strip() for c in mapping.columns]
mapping = mapping.dropna(subset=["Vulnerability category"]).copy()

def base_category(name):
    return re.sub(r"\s*\(.*?\)\s*$", "", str(name)).strip()

mapping["cat_base"] = mapping["Vulnerability category"].map(base_category)
mapping["owasp_code"] = (mapping["OWASP Top 10:2025 (from final CWE)"]
                         .astype(str).str.extract(r"(A\d\d)")[0])

owasp_by_cat = {}
for _, r in mapping.iterrows():
    if pd.notna(r["owasp_code"]):
        owasp_by_cat.setdefault(r["cat_base"], r["owasp_code"])

print("OWASP lookup built for", len(owasp_by_cat), "categories")


OWASP lookup built for 19 categories


In [14]:
# Count TP and FN
tp_count = (df["Status"].astype(str).str.strip().str.upper() == "TP").sum()
fn_count = (df["Status"].astype(str).str.strip().str.upper() == "FN").sum()

print("True Positives (TP):", tp_count)
print("False Negatives (FN):", fn_count)
print("Total TP + FN:", tp_count + fn_count)

In [ ]:
# ============================================================
# LOAD AND PREPARE DATA  (document-driven; nothing hardcoded)
# ============================================================
df = pd.read_excel(MASTER_FILE, sheet_name=MASTER_SHEET).dropna(how="all")
df.columns = [str(c).strip() for c in df.columns]
df = df[df["Status"].notna()].copy()

for t in TOOLS + (["Ground Truth"] if "Ground Truth" in df.columns else []):
    df[t] = df[t].astype(str).str.strip().str.upper()

team = df["App ID"].astype(str).str.extract(r"Team (\d+)")[0]
df["app"] = df["Platform"].astype(str).str[0] + "-T" + team

df["type"]  = df["Standard Category"].map(clean_category)
df["owasp"] = df["type"].map(lambda c: owasp_by_cat.get(base_category(c)))
df["sev"]   = df["Final Severity"].astype(str).str.strip().replace({"nan": None})

df["is_merged"] = df["Status"].astype(str).str.strip().str.lower().str.startswith("merged")
df["verified"]  = df["Status"].isin(VERIFIED_STATUS)
df["any_tool"]  = df[TOOLS].eq("Y").any(axis=1)

miss_owasp = df[df["verified"] & df["owasp"].isna()]["type"].unique()
if len(miss_owasp):
    print("WARNING: no OWASP mapping for categories:", list(miss_owasp))
miss_sev = df[df["verified"] & df["sev"].isna()]["V-ID"].tolist()
if len(miss_sev):
    print("WARNING: verified rows with no Final Severity:", miss_sev)
bad_sev = set(df[df["verified"]]["sev"].dropna()) - set(SEVERITY_ORDER)
if bad_sev:
    print("WARNING: unexpected severity labels:", bad_sev)

v  = df[df["verified"]].copy()
vp = v.copy()
apps = list(dict.fromkeys(df.loc[df["app"].notna(), "app"]))
N_APPS = len(apps)
print(f"{len(v)} verified vulnerabilities across {v['type'].nunique()} categories, {N_APPS} apps")

os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), dpi=150, bbox_inches="tight")
    plt.close(fig)


In [ ]:

# ============================================================
# FIG 01  Deduplication / verification funnel at two levels
#   Left:  within-app deduplication (one weakness per app = 1)  -> used for metrics
#   Right: across-app deduplication (one weakness type = 1)     -> used for the profile
# ============================================================
tool_rows = df[df["any_tool"]]
inst = [len(tool_rows), int((df.Status == "TP").sum()), int((df.Status == "FP").sum()),
        int((df.Status == "FN").sum()), len(v)]
typ = [tool_rows["type"].nunique(),
       tool_rows[tool_rows.Status == "TP"]["type"].nunique(),
       tool_rows[tool_rows.Status == "FP"]["type"].nunique(),
       v[v.Status == "FN"]["type"].nunique(),
       v["type"].nunique()]
labels = ["Deduplicated tool findings", "Confirmed by manual verification",
          "Rejected as false positive", "Found only by manual test cases",
          "Verified vulnerabilities"]
colours = ["#6B7280", "#2E8B57", "#B0B0B0", "#5B8DEF", "#1F2937"]
 
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, vals, title in [(axes[0], inst, f"Per application (instances, {N_APPS} apps)"),
                        (axes[1], typ, "Across applications (distinct types)")]:
    ax.barh(labels[::-1], vals[::-1], color=colours[::-1])
    for i, x in enumerate(vals[::-1]):
        ax.text(x + max(vals) * 0.02, i, str(x), va="center", fontweight="bold")
    ax.set_xlim(0, max(vals) * 1.18)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.set_title(title, loc="left", fontsize=11)
fig.suptitle(f"{inst[4]} verified vulnerability instances, representing {typ[4]} distinct weakness types",
             x=0.01, ha="left", fontweight="bold")
overlap = set(tool_rows[tool_rows.Status == "TP"]["type"]) & set(tool_rows[tool_rows.Status == "FP"]["type"])
if overlap:
    fig.text(0.01, -0.04, "Confirmed in some apps and rejected in others: " + ", ".join(sorted(overlap)),
             fontsize=9, style="italic")
save(fig, "fig01_funnel.png")
print("funnel instances", inst, "| types", typ)


In [ ]:
TEST_FILE ="Manual_Test_Pass_Fail_file"

### FIG T1  RO1 pass / fail matrix of all test cases

In [17]:
# ============================================================
# FIG T1  Pass / fail matrix of all core and conditional test cases (no title)
#   Green = Pass, red = Fail, grey = N/A, white = not filled in yet.
# ============================================================
tr = pd.read_excel(TEST_FILE, sheet_name="Test results", keep_default_na=False, na_values=[""])  # keep "N/A" as text
tr = tr[tr["Test ID"].astype(str).str.match(r"^(CORE|COND)-\d+")].copy()   # drops the summary rows
app_cols = [c for c in tr.columns if re.match(r"^[BL]-T\d+$", str(c))]
code = {"Pass": 2, "Fail": 0, "N/A": 1}
G = tr[app_cols].apply(lambda col: col.astype(str).str.strip().map(code)).astype(float).values
labels = (tr["Test ID"] + "  " + tr["Test name"].astype(str).str.slice(0, 45)).tolist()

cmap = matplotlib.colors.ListedColormap(["#E4572E", "#D9D9D9", "#2E8B57"])
cmap.set_bad("white")
fig, ax = plt.subplots(figsize=(13, 0.42 * len(tr) + 2))
ax.imshow(np.ma.masked_invalid(G), cmap=cmap, vmin=0, vmax=2, aspect="auto")
SYM = {0: "✗", 1: "–", 2: "✓"}
for i in range(G.shape[0]):
    for j in range(G.shape[1]):
        if not np.isnan(G[i, j]):
            ax.text(j, i, SYM[int(G[i, j])], ha="center", va="center", fontsize=10,
                    color="white" if G[i, j] != 1 else "#4B5563", fontweight="bold")
    vals = G[i][~np.isnan(G[i])]
    applicable = int((vals != 1).sum())
    passed = int((vals == 2).sum())
    ax.text(len(app_cols) - 0.3, i, f"  {passed}/{applicable} passed" if applicable else "  –",
            va="center", fontsize=9, color="#1F2937")
ax.set_xticks(range(len(app_cols)))
ax.set_xticklabels(app_cols, rotation=60, fontsize=9)
for lab in ax.get_xticklabels():
    lab.set_color(BOLT_COLOUR if lab.get_text().startswith("B") else LOVABLE_COLOUR)
ax.xaxis.tick_top()
ax.set_yticks(range(len(tr)))
ax.set_yticklabels(labels, fontsize=9)
n_core = int(tr["Test ID"].str.startswith("CORE").sum())
ax.axhline(n_core - 0.5, color="black", lw=1.5)
n_b = sum(c.startswith("B") for c in app_cols)
ax.axvline(n_b - 0.5, color="black", lw=1.5)
ax.set_xticks(np.arange(-.5, len(app_cols)), minor=True)
ax.set_yticks(np.arange(-.5, len(tr)), minor=True)
ax.grid(which="minor", color="white", lw=1.5)
ax.tick_params(which="both", length=0)
for s_ in ax.spines.values():
    s_.set_visible(False)
handles = [matplotlib.patches.Patch(color=c, label=l) for c, l in
           [("#2E8B57", "✓ Pass"), ("#E4572E", "✗ Fail"), ("#D9D9D9", "– N/A (feature absent or untestable)")]]
ax.legend(handles=handles, frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.45, -0.02), fontsize=9.5)
save(fig, "fig_t1_test_matrix.png")
blank = int(np.isnan(G).sum())
if blank:
    print(f"Note: {blank} cells are still empty in {TEST_FILE}; they are shown white.")

In [18]:
# ============================================================
# FIG 02  RO1  Distinct weakness types per OWASP category, by severity
#   Each type counted ONCE (deduplicated across apps).
#   Label = number of apps (of 18) with at least one weakness in that category.
# ============================================================
types_df = vp.groupby("type").agg(owasp=("owasp", "first"), sev=("sev", "first")).reset_index()
ct = pd.crosstab(types_df["owasp"], types_df["sev"]).reindex(columns=SEVERITY_ORDER, fill_value=0)
sev_rank = {s: i for i, s in enumerate(SEVERITY_ORDER)}
ct = ct.loc[sorted(ct.index, key=lambda o: (-ct.loc[o].sum(), -ct.loc[o, "Critical"]))]
apps_hit = vp.groupby("owasp")["app"].nunique()
 
fig, ax = plt.subplots(figsize=(10, 5.2))
bottom = np.zeros(len(ct))
xlabels = [OWASP_NAMES[o] for o in ct.index]
for s_ in SEVERITY_ORDER:
    ax.bar(xlabels, ct[s_], bottom=bottom, color=SEVERITY_COLOURS[s_], label=s_)
    bottom += ct[s_].values
for i, o in enumerate(ct.index):
    ax.text(i, bottom[i] + 0.1, f"{apps_hit[o]}/{N_APPS} apps", ha="center", va="bottom",
            fontsize=9, fontweight="bold")
ax.set_ylabel("Distinct verified weakness types")
ax.set_ylim(0, bottom.max() + 1.2)
ax.yaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
plt.xticks(rotation=25, ha="right")
ax.legend(title="Severity", frameon=False)
top = ct.index[0]
ax.set_title(f"{OWASP_NAMES[top][4:]} has the most weakness types ({int(ct.loc[top].sum())}) "
             f"and {int(ct.loc[top, 'Critical'])} of the {int(ct['Critical'].sum())} Critical ones",
             loc="left", fontweight="bold")
fig.text(0.01, -0.02, f"{len(types_df)} distinct weakness types. Label = applications with at least "
         "one weakness in the category.", fontsize=9, style="italic")
save(fig, "fig02_ro1_owasp_severity.png")
 
# ============================================================
# FIG 02b  RO1  Vulnerability categories grouped by OWASP, coloured by severity
#   One bar per vulnerability category (deduplicated across apps).
#   Bar length = apps affected (of 18). Groups ordered A01..A10.
# ============================================================
cat_df = vp.groupby("type").agg(owasp=("owasp", "first"), sev=("sev", "first"),
                                apps=("app", "nunique")).reset_index()
cat_df["sev_rank"] = cat_df["sev"].map({s_: i for i, s_ in enumerate(SEVERITY_ORDER)})
cat_df = cat_df.sort_values(["owasp", "apps", "sev_rank"], ascending=[True, False, True])
 
# Build rows: one bold header row per OWASP group, then its categories
rows_y, rows_lab, rows_hdr, y = [], [], [], 0
bar_y = {}
for o, g in cat_df.groupby("owasp", sort=True):
    rows_y.append(y); rows_lab.append(OWASP_NAMES[o]); rows_hdr.append(True); y += 1
    for _, r in g.iterrows():
        bar_y[r["type"]] = y
        rows_y.append(y); rows_lab.append("   " + r["type"]); rows_hdr.append(False); y += 1
    y += 0.4
cat_df["y"] = cat_df["type"].map(bar_y)
 
fig, ax = plt.subplots(figsize=(11, 0.36 * len(rows_y) + 1.5))
ax.barh(cat_df["y"], cat_df["apps"], color=[SEVERITY_COLOURS[s_] for s_ in cat_df["sev"]], height=0.75)
for _, r in cat_df.iterrows():
    ax.text(r["apps"] + 0.2, r["y"], f"{r['apps']}/{N_APPS}", va="center", fontsize=9)
ax.set_yticks(rows_y)
ax.set_yticklabels(rows_lab, fontsize=9.5)
for lab, hdr in zip(ax.get_yticklabels(), rows_hdr):
    if hdr:
        lab.set_fontweight("bold"); lab.set_color("#111827")
ax.tick_params(axis="y", length=0)
ax.invert_yaxis()
ax.set_xlim(0, N_APPS + 1.5)
ax.set_xticks(range(0, N_APPS + 1, 3))
ax.set_xlabel(f"Applications affected (of {N_APPS})")
handles = [matplotlib.patches.Patch(color=SEVERITY_COLOURS[s_], label=s_) for s_ in SEVERITY_ORDER]
ax.legend(handles=handles, title="Severity", frameon=False, ncol=4,
          loc="lower center", bbox_to_anchor=(0.4, 1.0))
fig.suptitle("Broken Access Control spans the most verified vulnerability categories",
             x=0.01, ha="left", fontweight="bold")
save(fig, "fig02b_ro1_categories_by_owasp.png")

In [19]:
 
# ============================================================
# FIG 03  RO1  Prevalence: % of apps affected by each type
# ============================================================
prev = (vp.groupby("type")["app"].nunique() / N_APPS).sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(prev.index, prev.values * 100, color="#4B5563")
for i, x in enumerate(prev.values):
    ax.text(x * 100 + 1, i, f"{int(round(x * N_APPS))}/{N_APPS}", va="center", fontsize=9)
ax.set_xlabel("% of applications affected")
ax.set_xlim(0, 115)
n15 = int((prev * N_APPS >= 15).sum())
ax.set_title(f"{n15} weaknesses appeared in 15 or more of the {N_APPS} applications",
             loc="left", fontweight="bold")
save(fig, "fig03_ro1_prevalence.png")
 

In [20]:
 
# ============================================================
# FIG 04  RO1  App x vulnerability type matrix (present / absent)
# ============================================================
types = prev.index[::-1]
M = pd.DataFrame(0, index=types, columns=apps)
for _, r in vp.iterrows():
    M.loc[r["type"], r["app"]] = 1
 
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(M.values, cmap=matplotlib.colors.ListedColormap(["#EEF2F5", "#C0392B"]), aspect="auto")
ax.set_xticks(range(N_APPS))
ax.set_xticklabels(apps, rotation=60, fontsize=9)
ax.set_yticks(range(len(types)))
ax.set_yticklabels(types, fontsize=9)
for i, a in enumerate(apps):
    ax.get_xticklabels()[i].set_color(BOLT_COLOUR if a.startswith("B") else LOVABLE_COLOUR)
n_bolt = sum(a.startswith("B") for a in apps)
ax.axvline(n_bolt - 0.5, color="black", lw=1.5)
ax.set_xticks(np.arange(-.5, N_APPS), minor=True)
ax.set_yticks(np.arange(-.5, len(types)), minor=True)
ax.grid(which="minor", color="white", lw=1.5)
ax.tick_params(which="minor", length=0)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("Every application shares the same core set of weaknesses (red = present)",
             loc="left", fontweight="bold")
save(fig, "fig04_ro1_app_matrix.png")

In [ ]:

# ============================================================
# FIG 05  RO1  Saturation curve per platform
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4.5))
for prefix, colour, label in [("B", BOLT_COLOUR, "Bolt.new"), ("L", LOVABLE_COLOUR, "Lovable")]:
    seen, ys = set(), []
    for a in [x for x in apps if x.startswith(prefix)]:
        seen |= set(vp[vp["app"] == a]["type"])
        ys.append(len(seen))
    ax.plot(range(1, len(ys) + 1), ys, marker="o", color=colour, label=label, lw=2.2)
    print(f"saturation {label}: {ys}")
ax.set_xlabel("Total Applications")
ax.set_ylabel("Distinct vulnerability types found")
ax.set_ylim(0)
ax.legend(frameon=False)
ax.set_title("New applications stopped revealing new vulnerability types",
             loc="left", fontweight="bold")
save(fig, "fig05_ro1_saturation.png")

In [ ]:
 
# ============================================================
# RO2 METRICS
#   TP = verified finding the tool flagged
#   FP = rejected finding the tool flagged
#   FN = verified finding the tool did NOT flag
# ============================================================
rows = []
for tool in TOOLS + ["ALL"]:
    flagged = df["any_tool"] if tool == "ALL" else df[tool].eq("Y")
    TP = int((flagged & (df.Status == "TP")).sum())
    FP = int((flagged & (df.Status == "FP")).sum())
    FN = int((~flagged & df["verified"]).sum())
    tp_rows = df[flagged & (df.Status == "TP")]
    rows.append({
        "tool": "All tools combined" if tool == "ALL" else TOOL_LABELS[tool],
        "TP": TP, "FP": FP, "FN": FN,
        "precision": TP / (TP + FP) if TP + FP else np.nan,
        "recall": TP / (TP + FN) if TP + FN else np.nan,
        "apps_with_verified_find": tp_rows["app"].nunique(),
        "owasp_coverage": tp_rows["owasp"].nunique() / v["owasp"].nunique(),
    })
metrics = pd.DataFrame(rows)
print("\nRO2 metrics\n", metrics.round(3).to_string(index=False))

In [ ]:
fp = df[df["Status"] == "FP"].copy()
fp["Flagged by"] = fp[TOOLS].eq("Y").apply(
    lambda r: ", ".join(TOOL_LABELS[t] for t in TOOLS if r[t]), axis=1)
fp["Why rejected"] = fp["What was found"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 1) Summary: one row per false-positive type
fp_summary = (fp.groupby("type")
                .agg(Flagged_by=("Flagged by", lambda s: ", ".join(sorted(set(", ".join(s).split(", "))))),
                     Apps=("app", "nunique"),
                     Instances=("app", "size"),
                     Why_rejected=("Why rejected", "first"))
                .sort_values("Instances", ascending=False)
                .reset_index()
                .rename(columns={"type": "False-positive type", "Flagged_by": "Flagged by",
                                 "Why_rejected": "Why rejected (example)"}))
print(f"{len(fp)} false positives, {fp['type'].nunique()} types, in {fp['app'].nunique()} apps\n")
print(fp_summary[["False-positive type", "Flagged by", "Apps", "Instances"]].to_string(index=False))

# 2) False positives per tool
print("\nFalse positives per tool:")
for t in TOOLS:
    print(f"  {TOOL_LABELS[t]:20s} {int(fp[t].eq('Y').sum())}")

# 3) Full list saved to Excel for the appendix
fp_list = fp[["app", "Platform", "type", "Flagged by", "Why rejected"]].rename(
    columns={"app": "App ID", "type": "False-positive type"}).sort_values(["False-positive type", "App ID"])
with pd.ExcelWriter(os.path.join(OUT_DIR, "false_positives.xlsx")) as w:
    fp_summary.to_excel(w, sheet_name="Summary", index=False)
    fp_list.to_excel(w, sheet_name="All false positives", index=False)

In [24]:

# ============================================================
# FIG 06  RO2  Precision and recall per tool
# ============================================================
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics))
w = 0.38
b1 = ax.bar(x - w / 2, metrics["precision"] * 100, w, color="#2E8B57", label="Precision")
b2 = ax.bar(x + w / 2, metrics["recall"] * 100, w, color="#5B8DEF", label="Recall")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5, f"{b.get_height():.0f}%",
            ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metrics["tool"], fontsize=10)
ax.get_xticklabels()[-1].set_fontweight("bold")
ax.set_ylim(0, 110)
ax.set_ylabel("%")
ax.legend(frameon=False)
all_recall = metrics.iloc[-1]["recall"]
ax.set_title(f"Even all four tools combined caught only {all_recall:.0%} of verified vulnerabilities",
             loc="left", fontweight="bold")
save(fig, "fig06_ro2_precision_recall.png")

In [ ]:
# ============================================================
# FIG 07  RO2  Coverage heatmap: tool x OWASP category
# ============================================================
cats = sorted(v["owasp"].unique())
H = pd.DataFrame(0, index=[TOOL_LABELS[t] for t in TOOLS] + ["Verified total"], columns=cats)
for t in TOOLS:
    d = df[df[t].eq("Y") & (df.Status == "TP")]
    for o in cats:
        H.loc[TOOL_LABELS[t], o] = int((d["owasp"] == o).sum())
for o in cats:
    H.loc["Verified total", o] = int((v["owasp"] == o).sum())
R = H.div(H.loc["Verified total"], axis=1)                 # share detected, for shading
 
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.imshow(R.values, cmap="Greens", vmin=0, vmax=1, aspect="auto")
for i in range(len(H)):
    for j in range(len(cats)):
        val = H.iloc[i, j]
        ax.text(j, i, str(val) if val else "–", ha="center", va="center",
                color="white" if R.iloc[i, j] > 0.6 else "black",
                fontweight="bold" if i == len(H) - 1 else "normal")
ax.set_xticks(range(len(cats)))
ax.set_xticklabels([OWASP_NAMES[o] for o in cats], rotation=25, ha="right", fontsize=9)
ax.set_yticks(range(len(H)))
ax.set_yticklabels(H.index)
ax.axhline(len(H) - 1.5, color="black", lw=1)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("No tool detected a single logging or exception-handling weakness",
             loc="left", fontweight="bold")
save(fig, "fig07_ro2_coverage.png")
 
# ============================================================
# FIG 07b  RO2  Coverage by vulnerability category (tool x category)
#   Cell = apps where the tool detected that category / apps affected.
#   Category coverage = categories detected at least once / all categories.
# ============================================================
cats_df = vp.groupby("type").agg(owasp=("owasp", "first"), apps=("app", "nunique")).reset_index()
cats_df = cats_df.sort_values(["owasp", "apps"], ascending=[True, False])
cols = [TOOL_LABELS[t] for t in TOOLS] + ["Any tool"]
C = pd.DataFrame(0, index=cats_df["type"], columns=cols)
for t in TOOLS:
    d = vp[vp[t].eq("Y") & (vp.Status == "TP")]
    for c_ in C.index:
        C.loc[c_, TOOL_LABELS[t]] = d[d["type"] == c_]["app"].nunique()
anyd = vp[vp.Status == "TP"]
for c_ in C.index:
    C.loc[c_, "Any tool"] = anyd[anyd["type"] == c_]["app"].nunique()
affected = cats_df.set_index("type")["apps"]
share = C.div(affected, axis=0)
 
# insert OWASP header rows
rows_lab, rows_hdr, data_rows = [], [], []
for o, g in cats_df.groupby("owasp", sort=True):
    rows_lab.append(OWASP_NAMES[o]); rows_hdr.append(True); data_rows.append(None)
    for c_ in g["type"]:
        rows_lab.append("   " + c_ + f"  ({affected[c_]} app{'s' if affected[c_] != 1 else ''})"); rows_hdr.append(False); data_rows.append(c_)
 
grid = np.full((len(rows_lab), len(cols)), np.nan)
for i, c_ in enumerate(data_rows):
    if c_ is not None:
        grid[i] = share.loc[c_].values
cmap = matplotlib.colormaps["Greens"].copy()
cmap.set_bad("white")
fig, ax = plt.subplots(figsize=(11, 0.36 * len(rows_lab) + 1.8))
ax.imshow(np.ma.masked_invalid(grid), cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i, c_ in enumerate(data_rows):
    if c_ is None:
        continue
    for j, col in enumerate(cols):
        n = C.loc[c_, col]
        ax.text(j, i, f"{n}/{affected[c_]}" if n else "–", ha="center", va="center", fontsize=8.5,
                color="white" if share.loc[c_, col] > 0.6 else "black",
                fontweight="bold" if col == "Any tool" else "normal")
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(cols, fontsize=9.5)
ax.xaxis.tick_top()
ax.set_yticks(range(len(rows_lab)))
ax.set_yticklabels(rows_lab, fontsize=9)
for lab, hdr in zip(ax.get_yticklabels(), rows_hdr):
    if hdr:
        lab.set_fontweight("bold")
ax.axvline(len(cols) - 1.5, color="black", lw=1)
ax.tick_params(length=0)
for s_ in ax.spines.values():
    s_.set_visible(False)
n_cat = len(C)
missed = int((C["Any tool"] == 0).sum())
fig.suptitle(f"{missed} of {n_cat} vulnerability categories were never detected by any tool",
             x=0.01, ha="left", fontweight="bold")
fig.text(0.01, -0.01, "Cell = applications where the tool detected the category / applications affected.",
         fontsize=9, style="italic")
save(fig, "fig07b_ro2_coverage_by_category.png")
cat_cov = (C > 0).sum() / n_cat
print("\nCategory coverage (categories detected at least once / all categories)")
print(cat_cov.round(3).to_string())

In [ ]:
# ============================================================
# FIG 08  RO2  Verified vulnerabilities by detection source (no title; add on slide)
#   Horizontal stacked bars per OWASP category + an "All categories" total bar.
# ============================================================
TOOL_C, MANUAL_C = "#2A9D8F", "#E76F51"
vp["source"] = np.where(vp.Status == "TP", "tool", "manual")
S = pd.crosstab(vp["owasp"], vp["source"]).reindex(columns=["tool", "manual"], fill_value=0)
S = S.loc[S.sum(axis=1).sort_values(ascending=False).index]         # biggest at top
S.loc["ALL"] = S.sum()
names = [OWASP_NAMES[o] if o != "ALL" else "All categories" for o in S.index]
ypos = list(range(len(S) - 1)) + [len(S) - 1 + 0.8]                  # gap before total
 
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.barh(ypos, S["tool"], color=TOOL_C, label="Detected by at least one tool", height=0.7)
ax.barh(ypos, S["manual"], left=S["tool"], color=MANUAL_C,
        label="Found only by manual test cases", height=0.7)
total_max = S.sum(axis=1).max()
for y_, (o, r) in zip(ypos, S.iterrows()):
    for val, left in [(r["tool"], 0), (r["manual"], r["tool"])]:
        if val >= total_max * 0.04:
            ax.text(left + val / 2, y_, str(int(val)), ha="center", va="center",
                    color="white", fontsize=10, fontweight="bold")
    tot = r["tool"] + r["manual"]
    ax.text(tot + total_max * 0.01, y_, f"{r['manual'] / tot:.0%} manual only",
            va="center", fontsize=9.5, color=MANUAL_C, fontweight="bold")
ax.set_yticks(ypos)
ax.set_yticklabels(names, fontsize=10.5)
ax.get_yticklabels()[-1].set_fontweight("bold")
ax.invert_yaxis()
ax.set_xlim(0, total_max * 1.22)
ax.set_xlabel(f"Verified vulnerability instances (across {N_APPS} apps)")
ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.45, 1.0), fontsize=10)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.axhline(len(S) - 1.1, color="#D1D5DB", lw=1)
save(fig, "fig08_ro2_detection_source.png")

In [ ]:
# ============================================================
# FIG 09  RO4  Bolt.new vs Lovable by vulnerability category (butterfly chart)
#   Left = Bolt.new apps affected (of 9), right = Lovable apps affected (of 9).
#   Grouped into: found on both platforms / Bolt.new only / Lovable only.
#   No title; add it on the slide.
# ============================================================
per_app = vp.groupby(["Platform", "app"]).size()
bolt_counts = per_app["Bolt.new"].values
lov_counts = per_app["Lovable"].values
n_b = sum(a.startswith("B") for a in apps)          # number of Bolt.new apps
n_l = sum(a.startswith("L") for a in apps)
 
P = vp.groupby(["type", "Platform"])["app"].nunique().unstack(fill_value=0)
for col in ["Bolt.new", "Lovable"]:
    if col not in P:
        P[col] = 0
P["group"] = np.where((P["Bolt.new"] > 0) & (P["Lovable"] > 0), "Found on both platforms",
             np.where(P["Bolt.new"] > 0, "Bolt.new only", "Lovable only"))
group_order = ["Found on both platforms", "Bolt.new only", "Lovable only"]
 
rows = []                                    # (label, is_header, bolt, lovable)
for gname in group_order:
    g = P[P["group"] == gname].copy()
    if g.empty:
        continue
    g["tot"] = g["Bolt.new"] + g["Lovable"]
    g = g.sort_values(["tot", "Bolt.new"], ascending=False)
    rows.append((f"{gname}  ({len(g)})", True, 0, 0))
    for t, r in g.iterrows():
        rows.append((t, False, int(r["Bolt.new"]), int(r["Lovable"])))
 
GAP = 8.5                                     # half-width of the centre label column
fig, ax = plt.subplots(figsize=(13, 0.42 * len(rows) + 1.6))
for i, (lab, hdr, bc, lc) in enumerate(rows):
    if hdr:
        ax.text(0, i, lab, ha="center", va="center", fontsize=10.5, fontweight="bold", color="#111827")
        if i > 0:
            ax.axhline(i - 0.6, color="#D1D5DB", lw=1)
        continue
    # faint track showing the maximum (all apps)
    ax.barh(i, -n_b, left=-GAP, color="#F1F3F5", height=0.62)
    ax.barh(i, n_l, left=GAP, color="#F1F3F5", height=0.62)
    if bc:
        ax.barh(i, -bc, left=-GAP, color=BOLT_COLOUR, height=0.62)
        ax.text(-GAP - bc - 0.25, i, f"{bc}/{n_b}", ha="right", va="center", fontsize=9, color=BOLT_COLOUR,
                fontweight="bold")
    if lc:
        ax.barh(i, lc, left=GAP, color=LOVABLE_COLOUR, height=0.62)
        ax.text(GAP + lc + 0.25, i, f"{lc}/{n_l}", ha="left", va="center", fontsize=9, color=LOVABLE_COLOUR,
                fontweight="bold")
    ax.text(0, i, lab, ha="center", va="center", fontsize=9.5, color="#1F2937")
 
ax.text(-GAP - n_b / 2, -1.3, f"Bolt.new  ({n_b} apps)", ha="center", va="center",
        fontsize=12, fontweight="bold", color=BOLT_COLOUR)
ax.text(GAP + n_l / 2, -1.3, f"Lovable  ({n_l} apps)", ha="center", va="center",
        fontsize=12, fontweight="bold", color=LOVABLE_COLOUR)
ax.set_xlim(-GAP - n_b - 1.6, GAP + n_l + 1.6)
ax.set_ylim(len(rows) - 0.4, -1.9)
ax.axis("off")
fig.text(0.5, 0.005, "Bar = applications on that platform with the vulnerability category "
         f"(grey track = all {n_b} apps).", ha="center", fontsize=9, style="italic")
save(fig, "fig09_ro4_platform_comparison.png")

In [ ]:
 
# ============================================================
# RO4 NUMBERS (printed for your table)
# ============================================================
mw = mannwhitneyu(bolt_counts, lov_counts)
print("\nRO4")
print(f"Mean per app  Bolt.new {bolt_counts.mean():.1f} | Lovable {lov_counts.mean():.1f}")
print(f"Median        Bolt.new {np.median(bolt_counts):.0f} | Lovable {np.median(lov_counts):.0f}")
print(f"Mann-Whitney U = {mw.statistic:.1f}, p = {mw.pvalue:.3f}")
print("Severity by platform:\n", pd.crosstab(v["Platform"], v["sev"]).reindex(columns=SEVERITY_ORDER, fill_value=0))
for p in ["Bolt.new", "Lovable"]:
    d = df[df["Platform"] == p]
    f = d["Scanner"].eq("Y")
    TP = int((f & (d.Status == "TP")).sum())
    FP = int((f & (d.Status == "FP")).sum())
    FN = int((~f & d["verified"]).sum())
    prec = TP / (TP + FP) if TP + FP else float("nan")
    print(f"{p} scanner: TP={TP} FP={FP} FN={FN} precision={prec:.0%} recall={TP / (TP + FN):.0%}")

In [ ]:

# ============================================================
# FIG 09  RO4  Bolt.new vs Lovable by OWASP category (diverging)
# ============================================================
D = pd.crosstab(v["owasp"], v["Platform"]).reindex(ct.index[::-1]).fillna(0).astype(int)
per_app = v.groupby(["Platform", "app"]).size()
bolt_counts = per_app["Bolt.new"].values
lov_counts = per_app["Lovable"].values
 
fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(D))
ax.barh(y, -D["Bolt.new"], color=BOLT_COLOUR, label="Bolt.new")
ax.barh(y, D["Lovable"], color=LOVABLE_COLOUR, label="Lovable")
for i, (b, l) in enumerate(zip(D["Bolt.new"], D["Lovable"])):
    ax.text(-b - 1, i, str(b), ha="right", va="center")
    ax.text(l + 1, i, str(l), ha="left", va="center")
ax.set_yticks(y)
ax.set_yticklabels([OWASP_NAMES[o] for o in D.index])
ax.axvline(0, color="black", lw=1)
lim = D.values.max() + 8
ax.set_xlim(-lim, lim)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.legend(frameon=False, loc="lower right")
ax.set_title(f"Both platforms had a similar profile: {bolt_counts.mean():.1f} vs "
             f"{lov_counts.mean():.1f} verified vulnerabilities per app",
             loc="left", fontweight="bold")
save(fig, "fig09_ro4_platform_owasp.png")
 